In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import string
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [4]:
df.head(3)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [5]:
df.shape

(2000, 8)

In [6]:
# Q1
freq = df['answer'].value_counts()
most_freq_count  = freq.max() 
least_freq_count = freq.min()

print(f"Q1 Answer: {most_freq_count + least_freq_count}")
 

Q1 Answer: 814


In [7]:
# Q2
def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text
 
df['clean_prompt'] = df['prompt'].apply(clean_text)
 
all_words = []
for text in df['clean_prompt']:
    all_words.extend(text.split())
 
ans_q2 = len(set(all_words))
print(f"Q2 Answer: {ans_q2}")

Q2 Answer: 859


In [8]:
# Q3
row1_clean = df[df['id'] == 1]['clean_prompt'].values[0]
words_row1 = row1_clean.split()
filtered   = [w for w in words_row1 if w not in ENGLISH_STOP_WORDS]
ans_q3 = len(filtered)
print(f"Q3 Answer: {ans_q3}") 

Q3 Answer: 13


In [9]:
# Q4
combined_texts = []
for _, row in df.iterrows():
    combined_texts.append(str(row['prompt']))
    for opt in ['A', 'B', 'C', 'D', 'E']:
        combined_texts.append(str(row[opt]))
 
tfidf = TfidfVectorizer(stop_words='english')
tfidf.fit(combined_texts)
ans_q4 = len(tfidf.vocabulary_)
print(f"Q4 Answer: {ans_q4}")

Q4 Answer: 2762


In [10]:
# Q5
row1 = df[df['id'] == 1].iloc[0]
vec_prompt = tfidf.transform([row1['prompt']])
vec_optA= tfidf.transform([row1['A']])
sim= cosine_similarity(vec_prompt, vec_optA)[0][0]
ans_q5 = round(sim, 4)
print(f"Q5 Answer: {ans_q5}") 

Q5 Answer: 0.2328


In [11]:
# Q6
options = ['A', 'B', 'C', 'D', 'E']
correct = 0
for _, row in df.iterrows():
    prompt_vec=tfidf.transform([row['prompt']])
    sims = {opt: cosine_similarity(prompt_vec, tfidf.transform([str(row[opt])]))[0][0]
            for opt in options}
    if max(sims, key=sims.get) == row['answer']:
        correct += 1
 
ans_q6 =round(correct / len(df) * 100, 2)
print(f"Q6 Answer: {ans_q6}%") 

Q6 Answer: 13.7%


In [12]:
def map_at_3(ground_truth, predictions):
    score =0.0
    hits= 0
    for i, pred in enumerate(predictions[:3]):
        if pred == ground_truth:
            hits  += 1
            score += hits / (i + 1)
    return score

In [13]:
# Q7
ans_q7 =map_at_3('C', ['C', 'A', 'B'])
print(f"Q7 Answer: {ans_q7}")

Q7 Answer: 1.0


In [14]:
# Q8
ans_q8=map_at_3('B', ['D', 'B', 'E'])
print(f"Q8 Answer: {ans_q8}")

Q8 Answer: 0.5


In [15]:
# Q9
top3_majority = df['answer'].value_counts().index.tolist()[:3]  # ['B','C','A']
scores_q9= [map_at_3(row['answer'], top3_majority) for _, row in df.iterrows()]
ans_q9=round(np.mean(scores_q9), 4)
print(f"Q9 Answer: {ans_q9}") 

Q9 Answer: 0.4212


In [16]:
# Q10
scores_q10 = []
for _, row in df.iterrows():
    prompt_vec  = tfidf.transform([row['prompt']])
    sims= {opt: cosine_similarity(prompt_vec, tfidf.transform([str(row[opt])]))[0][0]
                   for opt in options}
    top3_preds= sorted(sims, key=sims.get, reverse=True)[:3]
    scores_q10.append(map_at_3(row['answer'], top3_preds))
 
ans_q10 = round(np.mean(scores_q10), 4)
print(f"Q10 Answer: {ans_q10}")

Q10 Answer: 0.3119
